In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from typing import List
from langchain_chroma import Chroma

/home/kapil/rag-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = PyPDFLoader("https://www.gita-society.com/bhagavad-gita-in-english-source-file.pdf")
# loader = PyPDFLoader("https://www.prabhupada-books.de/pdf/Bhagavad-gita-As-It-Is.pdf")

document = loader.load()

In [4]:
document

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2021-08-05T09:35:03-07:00', 'title': 'Bhagavad-Gita verses in simple english Language', 'author': 'Ram', 'moddate': '2021-08-05T09:35:03-07:00', 'source': 'https://www.gita-society.com/bhagavad-gita-in-english-source-file.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1'}, page_content='BHAGAVAD-GITA in ENGLISH (Source) \nFor commentaries: \nhttps://www.gita-society.com/Read-bhagavad-gita.html \n \nCONTENTS \nINTRODUCTION ...................................................... 1 \n1. Arjuna’s Dilemma .................................................. 3 \n2. Spiritual knowledge................................................ 3 \nThe spirit is eternal, body is transitory ........................ 4 \nDeath and Reincarnation of the soul .......................... 4 \nDuty of a warrior ......................................................... 5 \nImportance of Karma-yoga, the selfl

In [5]:
from typing import Any

class EmbeddingManager:
    def __init__(self, model_name:'str'):
        """
        Intialze tne embedding manager
        
        Args: 
            model_name: hugging face model for sentense enbedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """ Loader function to load the transformer Model """
        try: 
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding model loaded successfully with dimenstions: {self.model.get_embedding_dimension()}")
        except Exception as e: 
            print(f"Failed to load model {self.model_name}", e)
            raise  
          
    def generate_embeddings(self, text: List[str]):
        if not self.model: 
            raise ValueError("Model is not loaded")
        print(f"generating embeddings for {len(text)} text")
        embeddings = self.model.encode(text, show_progress_bar=False)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings  
    
    def document_chunker(self, document_chunk: Any):
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=100,
            chunk_overlap=0
            )
        return text_splitter.split_text(document_chunk)

In [6]:
import chromadb


class VectorStore:
    def __init__(self, collection_name: str, persistant_directory: str):
        self.collection_name = collection_name
        self.persistant_dirctory = self.persistant_dirctory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """ Initialize chroma DB client and collection store """
        try:
            # Create chroma DB client
            self.client = chromadb.PersistentClient(self.persistant_dirctory)
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={ "decripiotn": "PDF docuement enbedding for RAG" }
            )
            print(f"Vector store initialized collection: {self.collection_name}")
            print(f"Existing  documents in collection: {self.collection.count()}")

        except Exception as e:
                print(f"Error while initializing vector store {self.collection_name}", e)
                raise


In [7]:
class LangchainChroma: 
    def __init__(self, collection_name: str, persistant_dircetory: str):
        self.collection_name = collection_name
        self.persistant_directory = persistant_dircetory
        self._initialize_store()
                
    def _initialize_store(self):
        try:
            self.client = Chroma(
                collection_name=self.collection_name,
                embedding_function="all-MiniLM-L6-v2",
                persist_directory=self.persistant_directory
            )
        except Exception as e:
            print(f"Exception while initializing vector store", e)
            raise
    
    def add_documents_to_vector_store(self, document_embedding) -> bool:
        self.client.aadd_documents(document_embedding)
        # self.client.persist()
        return True
    
    def search_documents(self, query: str):
        print("Query", query)
        print(self.client)
        query_response = self.client.similarity_search(
            query, 
            3
        )
        
        print("query_response: ", query_response)
        # query = self.client.similarity_search_with_score(
        #     query=query,
        #     k=3
        # )

In [8]:
# Initialize embedding manager
embedding_model = "all-MiniLM-L6-v2"
embedding_manager = EmbeddingManager(embedding_model)
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


/home/kapil/rag-from-scratch/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3177.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully with dimenstions: 384


In [9]:
# Chunk the document
text_chunks = []
embeddings = []
for item in document:
    chunks = embedding_manager.document_chunker(item.page_content)
    item = {
        "text": chunks,
        "metas": item.metadata
    }
    text_chunks.append(item)

# Generate Embeddings
   
for chunk in text_chunks:
    embeddings.append(embedding_manager.generate_embeddings(chunk["text"]))    

generating embeddings for 34 text
Generated embeddings with shape: (34, 384)
generating embeddings for 42 text
Generated embeddings with shape: (42, 384)
generating embeddings for 42 text
Generated embeddings with shape: (42, 384)
generating embeddings for 28 text
Generated embeddings with shape: (28, 384)
generating embeddings for 35 text
Generated embeddings with shape: (35, 384)
generating embeddings for 44 text
Generated embeddings with shape: (44, 384)
generating embeddings for 28 text
Generated embeddings with shape: (28, 384)
generating embeddings for 45 text
Generated embeddings with shape: (45, 384)
generating embeddings for 41 text
Generated embeddings with shape: (41, 384)
generating embeddings for 42 text
Generated embeddings with shape: (42, 384)
generating embeddings for 41 text
Generated embeddings with shape: (41, 384)
generating embeddings for 41 text
Generated embeddings with shape: (41, 384)
generating embeddings for 44 text
Generated embeddings with shape: (44, 384)

In [10]:
len(embeddings)

53

In [11]:
collection_name = 'bhagwat_gita_plain_english'
persistant_directory = '/home/kapil/rag-from-scratch/vector_store'
vector_store = LangchainChroma(
    collection_name=collection_name,
    persistant_dircetory=persistant_directory,
)

In [12]:
#Adding document to vector db

resp = vector_store.add_documents_to_vector_store(embeddings)

if resp:
    print("Docuemnt added to vector store")

Docuemnt added to vector store


/tmp/ipykernel_350650/2379625741.py:19: RuntimeWarning: coroutine 'VectorStore.aadd_documents' was never awaited
  self.client.aadd_documents(document_embedding)


In [13]:
similar_docs = vector_store.search_documents("What is karma?")

Query What is karma?


AttributeError: 'str' object has no attribute 'embed_query'

In [ ]:
# Vector store Initialization
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function="all-MiniLM-L6-v2",
    persist_directory="/home/kapil/rag-from-scratch/vector_store"
)

InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [ ]:
ids = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,
      42,43,44,45,46,47,48,49,50,51,52]

In [ ]:
# Adding documents to vector db

# vector_store.add_documents(documents=text_chunks, id=ids)

In [ ]:
similar_docs = vector_store.similarity_search("What is yoga?")

AttributeError: 'str' object has no attribute 'embed_query'